### Dipole correction



[BROKEN LINK: index:dipole correction]

A subtle problem can arise when an adsorbate is placed on one side of
a slab with periodic boundary conditions, which is currently the
common practice.  The problem is that this gives the slab a dipole
moment. The array of dipole moments created by the periodic boundary
conditions generates an electric field that can distort the electron
density of the slab and change the energy. The existence of this field
in the vacuum also makes the zero-potential in the vacuum ill-defined,
thus the work function is not well-defined. One solution to this
problem is to use slabs with adsorbates on both sides, but then very
thick (eight to ten layers) slabs must be used to ensure the
adsorbates do not interact through the slab. An alternative solution,
the dipole correction scheme, was developed by Neugebauer and
Scheffler cite:PhysRevB.46.16067 and later corrected by Bengtsson
cite:PhysRevB.59.12301. In this technique, an external field is
imposed in the vacuum region that exactly cancels the artificial field
caused by the slab dipole moment. The advantage of this approach is
that thinner slabs with adsorbates on only one side can be used.

There are also literature reports that the correction is small
cite:morikawa2001:c2h2-si.  Nevertheless, in the literature the use
of this correction is fairly standard, and it is typical to at least
consider the correction.

Here we will just illustrate the effect.



#### Slab with no dipole correction



We simply run the calculation here, and compare the results later.



In [1]:
# compute local potential of slab with no dipole
from ase.lattice.surface import fcc111, add_adsorbate
from vasp import Vasp

slab = fcc111('Al', size=(2, 2, 2), vacuum=10.0)
add_adsorbate(slab, 'Na', height=1.2, position='fcc')
slab.center()

nodip_calc = Vasp(label='surfaces/Al-Na-nodip',
                  xc='PBE',
                  encut=340,
                  kpts=(2, 2, 1),
                  lcharg=True,
                  lvtot=True,
                  lvhar=True,
                  atoms=slab)

nodip_energy = nodip_calc.get_potential_energy()
print(f'Energy without dipole correction: {nodip_energy:.4f} eV')

/opt/conda/lib/python3.9/site-packages/ase/lattice/surface.py:17: UserWarning: Moved to ase.build
  warnings.warn('Moved to ase.build')


Energy without dipole correction: -22.5526 eV


    -22.55264459

![img](./images/Na-Al-slab.png "Example slab with a Na atom on it for illustrating the effects of dipole corrections.")



#### Slab with a dipole correction



Note this takes a considerably longer time to run than without a dipole correction! In VASP there are several levels of dipole correction to apply. You can use the [BROKEN LINK: incar:IDIPOL] tag to turn it on, and specify which direction to apply it in (1=$x$, 2=$y$, 3=$z$, 4=$(x,y,z)$). This simply corrects the total energy and forces. It does not change the contents of LOCPOT. For that, you have to also set the [BROKEN LINK: incar:LDIPOL] and [BROKEN LINK: incar:DIPOL] tags. It is not efficient to set all three at the same time for some reason. The VASP manual recommends you first set IDIPOL to get a converged electronic structure, and then set LDIPOL to True, and set the center of electron density in DIPOL. That makes these calculations a multistep process, because we must run a calculation, analyze the charge density to get the center of charge, and then run a second calculation.



In [2]:
# compute local potential with dipole calculation on
from ase.lattice.surface import fcc111, add_adsorbate
from vasp import Vasp
import numpy as np

slab_dip = fcc111('Al', size=(2, 2, 2), vacuum=10.0)
add_adsorbate(slab_dip, 'Na', height=1.2, position='fcc')
slab_dip.center()

# Step 1: Run with idipol to get converged electronic structure
dip_calc = Vasp(label='surfaces/Al-Na-dip',
                xc='PBE',
                encut=340,
                kpts=(2, 2, 1),
                lcharg=True,
                idipol=3,
                lvtot=True,
                lvhar=True,
                atoms=slab_dip)

e1 = dip_calc.get_potential_energy()
print(f'Step 1 energy: {e1:.4f} eV')

# Get charge density to find center
grid_points, cd = dip_calc.get_charge_density()

# Calculate electron density center
# cd is 3D array, grid_points gives the positions
n0, n1, n2 = cd.shape
uc = slab_dip.get_cell()

# Create coordinate grids
x = np.linspace(0, uc[0][0], n0)
y = np.linspace(0, uc[1][1], n1) 
z = np.linspace(0, uc[2][2], n2)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

voxel_volume = slab_dip.get_volume() / (n0 * n1 * n2)
total_charge = cd.sum() * voxel_volume

electron_density_center = np.array([
    (cd * X).sum() * voxel_volume / total_charge,
    (cd * Y).sum() * voxel_volume / total_charge,
    (cd * Z).sum() * voxel_volume / total_charge
])

print(f'electron-density center = {electron_density_center}')

# Get scaled coordinates
sedc = np.dot(np.linalg.inv(uc.T), electron_density_center.T).T
sedc = np.round(sedc, 4)

# Step 2: Run with dipole correction
dip_calc2 = Vasp(label='surfaces/Al-Na-dip-step2',
                 xc='PBE',
                 encut=340,
                 kpts=(2, 2, 1),
                 lcharg=True,
                 idipol=3,
                 ldipol=True,
                 dipol=list(sedc),
                 lvtot=True,
                 lvhar=True,
                 atoms=slab_dip)

dip_energy = dip_calc2.get_potential_energy()
print(f'Step 2 energy with dipole: {dip_energy:.4f} eV')

Step 1 energy: -22.5440 eV
electron-density center = [ 2.74280436  2.47533099 11.16704957]
Step 2 energy with dipole: -22.5461 eV


#### Comparing no dipole correction with a dipole correction



To see the difference in what the dipole correction does, we now plot the potentials from each calculation.



In [3]:
# compute local potential of slab with no dipole
from ase.lattice.surface import fcc111, add_adsorbate
from vasp import Vasp

slab = fcc111('Al', size=(2, 2, 2), vacuum=10.0)
add_adsorbate(slab, 'Na', height=1.2, position='fcc')
slab.center()

nodip_calc = Vasp(label='surfaces/Al-Na-nodip',
                  xc='PBE',
                  encut=340,
                  kpts=(2, 2, 1),
                  lcharg=True,
                  lvtot=True,
                  lvhar=True,
                  atoms=slab)

nodip_energy = nodip_calc.get_potential_energy()
print(f'Energy without dipole correction: {nodip_energy:.4f} eV')

Energy without dipole correction: -22.5526 eV


![img](./images/dip-vs-nodip-esp.png "Comparison of the electrostatic potentials with a dipole correction and without it.")

The key points to notice in this figure are:

1.  The two deep dips are where the atoms are.
2.  Without a dipole correction, the electrostatic potential never flattens out. there is near constant slope in the vacuum region, which means there is an electric field there.
3.  With a dipole correction the potential is flat in the vacuum region, except for the step jump near 23 &Aring;.
4.  The difference between the Fermi level and the flat vacuum potential is the work function.
5.  The difference in energy with and without the dipole correction here is small.

